## I will Work with Each table and in each Table each column to solve problems in it 

### Customers Table

In [1]:
from pyspark.sql.functions import * 
Customers_df=spark.read.table("Olist_Lakehouse.bronze.customers")
display(Customers_df)

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d5d1565b-fa08-43f6-adf0-40eddfa345a2)

In [43]:
Customers_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 45, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### If Customer_id has duplicates it will remove them 

In [44]:
Customers_df=Customers_df.dropDuplicates(["customer_id"])

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 46, Finished, Available, Finished, False)

#### If Customer_id has Nulls it will remove them 

In [45]:
Customers_df=Customers_df.filter(col("customer_id").isNotNull())

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 47, Finished, Available, Finished, False)

#### I will trim customer_id if it has extra spaces

In [46]:
Customers_df=Customers_df.withColumn("customer_id",trim("customer_id"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 48, Finished, Available, Finished, False)

#### After I trim customer_id we maybe have values like "" so we will drop them 

In [47]:
Customers_df=Customers_df.filter(col("customer_id") != "")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 49, Finished, Available, Finished, False)

#### all customer_id have to be its length 32 character so we have to keep only values with 32 characters

In [48]:
Customers_df=Customers_df.filter(length(col("customer_id")) == 32)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 50, Finished, Available, Finished, False)

### Column "Customer_unique_id"

#### customer could have duplicate values in customer_unique_id

#### But if it has Null values we will remove them 

In [49]:
Customers_df = Customers_df.filter(col("customer_unique_id").isNotNull())

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 51, Finished, Available, Finished, False)

#### Trim the column

In [50]:
Customers_df=Customers_df.withColumn("customer_unique_id",trim(col("customer_unique_id")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 52, Finished, Available, Finished, False)

#### I will drop values like ""

In [51]:
Customers_df=Customers_df.filter(col("customer_unique_id") != "")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 53, Finished, Available, Finished, False)

### Column zip_code 

#### First we will check data type 

In [52]:
Customers_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 54, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### I will replace null values with zero 

In [53]:
Customers_df=Customers_df.fillna({"customer_zip_code_prefix":0})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 55, Finished, Available, Finished, False)

#### i Will cast it to add leading zeros if it is not 5 digits

In [54]:
Customers_df=Customers_df.withColumn("customer_zip_code_prefix",col("customer_zip_code_prefix").cast("string"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 56, Finished, Available, Finished, False)

In [55]:
Customers_df=Customers_df.withColumn("customer_zip_code_prefix",lpad(col("customer_zip_code_prefix"),5,"0"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 57, Finished, Available, Finished, False)

### Column Customer City

#### I will replace null values with "unknown"

In [56]:
Customers_df=Customers_df.fillna({"customer_city":"unknown"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 58, Finished, Available, Finished, False)

#### Trim the column

In [57]:
Customers_df=Customers_df.withColumn("customer_city",trim(col("customer_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 59, Finished, Available, Finished, False)

#### When customer_city will be "" i will convert it to unknown

In [58]:
Customers_df=Customers_df.withColumn("customer_city",when(col("customer_city") == "","unknown").otherwise(col("customer_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 60, Finished, Available, Finished, False)

#### Replace multiple space to one space

In [59]:
Customers_df= Customers_df.withColumn("customer_city",regexp_replace(col("customer_city"),"  "," "))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 61, Finished, Available, Finished, False)

#### Make it lower case

In [60]:
Customers_df=Customers_df.withColumn("customer_city",lower(col("customer_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 62, Finished, Available, Finished, False)

#### if length of it less than 3 i will convert it to unknown

In [61]:
Customers_df=Customers_df.withColumn("customer_city",when(length(col("customer_city")) < 3,"unknown").otherwise(col("customer_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 63, Finished, Available, Finished, False)

### Column customer state 

#### Fill null values with UN

In [62]:
Customers_df=Customers_df.fillna({"customer_state":"UN"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 64, Finished, Available, Finished, False)

#### trim customer_state

In [63]:
Customers_df=Customers_df.withColumn("customer_state",trim(col("customer_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 65, Finished, Available, Finished, False)

#### When customer_state will be "" i will convert it to UN

In [64]:
Customers_df=Customers_df.withColumn("customer_state",when(col("customer_state") == "","UN").otherwise(col("customer_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 66, Finished, Available, Finished, False)

#### Make it upper CASE

In [65]:
Customers_df=Customers_df.withColumn("customer_state",upper(col("customer_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 67, Finished, Available, Finished, False)

#### Check the length it must be 2 otherwise it will be UN  

In [66]:
Customers_df=Customers_df.withColumn("customer_state",when(length(col("customer_state")) != 2,"UN").otherwise(col("customer_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 68, Finished, Available, Finished, False)

In [67]:
Customers_df = Customers_df.withColumn("silver_loaded_date", current_timestamp())
Customers_df = Customers_df.withColumn("silver_source_table", lit("bronze.customers"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 69, Finished, Available, Finished, False)

In [68]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
Customers_df.write.format("delta").mode("append").saveAsTable("silver.customers")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 70, Finished, Available, Finished, False)

### Now Products Table

In [112]:
Products_df=spark.read.table("Olist_Lakehouse.bronze.products")
display(Products_df)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 114, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77066160-1048-4965-a2ba-6afaaa12c812)

In [113]:
Products_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 115, Finished, Available, Finished, False)

root
 |-- Product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: double (nullable = true)
 |-- product_description_lenght: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### Product_id Same Rules as Customer_id

In [114]:
Products_df=Products_df.filter(col("Product_id").isNotNull())

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 116, Finished, Available, Finished, False)

In [115]:
Products_df=Products_df.dropDuplicates(["Product_id"])

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 117, Finished, Available, Finished, False)

In [116]:
Products_df=Products_df.withColumn("Product_id",trim(col("Product_id")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 118, Finished, Available, Finished, False)

In [117]:
Products_df=Products_df.filter(col("Product_id") != "")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 119, Finished, Available, Finished, False)

#### Column Category Name

#### if it is null i will make it unknown

In [118]:
Products_df=Products_df.fillna({"product_category_name":"unknown"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 120, Finished, Available, Finished, False)

#### I will Trim it  

In [119]:
Products_df = Products_df.withColumn("product_category_name",trim(col("product_category_name")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 121, Finished, Available, Finished, False)

#### After trim it if it is "" i will replace it with unknown 

In [121]:
Products_df=Products_df.withColumn("product_category_name",when(col("product_category_name") == "","unknwon").otherwise(col("product_category_name"))) 

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 123, Finished, Available, Finished, False)

#### Make it Lower Case

In [122]:
Products_df=Products_df.withColumn("product_category_name",lower(col("product_category_name")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 124, Finished, Available, Finished, False)

### Column product name length

#### First the column is the number of characters so it always will be integer

In [123]:
Products_df = Products_df.withColumnRenamed(
    "product_name_lenght",
    "product_name_length"
)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 125, Finished, Available, Finished, False)

In [124]:
Products_df=Products_df.withColumn("product_name_length",col("product_name_length").cast("int"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 126, Finished, Available, Finished, False)

#### Check Mean and median to fill null values

In [125]:
Products_df.select(
    mean(col("product_name_length")).alias("mean"),
    stddev(col("product_name_length")).alias("stddev"),
    min(col("product_name_length")).alias("min"),
    max(col("product_name_length")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_name_length", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 127, Finished, Available, Finished, False)

+-----------------+------------------+---+---+
|             mean|            stddev|min|max|
+-----------------+------------------+---+---+
|48.47694876472589|10.245740725237209|  5| 76|
+-----------------+------------------+---+---+

Median: 51.0


In [126]:
Products_df = Products_df.withColumn("product_name_length",
    when(col("product_name_length").isNull(), median_val )
    .otherwise(col("product_name_length")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 128, Finished, Available, Finished, False)

#### Column product description length

#### Same as product name length

In [127]:
Products_df = Products_df.withColumnRenamed(
    "product_description_lenght",
    "product_description_length"
)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 129, Finished, Available, Finished, False)

In [128]:
Products_df=Products_df.withColumn("product_description_length",col("product_description_length").cast("int"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 130, Finished, Available, Finished, False)

In [129]:
Products_df.select(
    mean(col("product_description_length")).alias("mean"),
    stddev(col("product_description_length")).alias("stddev"),
    min(col("product_description_length")).alias("min"),
    max(col("product_description_length")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_description_length", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 131, Finished, Available, Finished, False)

+-----------------+-----------------+---+----+
|             mean|           stddev|min| max|
+-----------------+-----------------+---+----+
|771.4952846232337|635.1152246349569|  4|3992|
+-----------------+-----------------+---+----+

Median: 594.0


In [130]:
Products_df = Products_df.withColumn("product_description_length",
    when(col("product_description_length").isNull(), median_val )
    .otherwise(col("product_description_length")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 132, Finished, Available, Finished, False)

### Column product photos qty

#### The quantity of photos can be null so i will convert null values to 0 and convert the column to integer 

In [131]:
Products_df = Products_df.withColumn("product_photos_qty",
    when(col("product_photos_qty").isNull(), 0)
    .when(col("product_photos_qty") < 0, 0)
    .otherwise(col("product_photos_qty").cast("int")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 133, Finished, Available, Finished, False)

### Column product weight g

In [132]:
Products_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 134, Finished, Available, Finished, False)

root
 |-- Product_id: string (nullable = true)
 |-- product_category_name: string (nullable = false)
 |-- product_name_length: double (nullable = true)
 |-- product_description_length: double (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### First i will replace 0 or less than 0 with null

In [133]:
Products_df=Products_df.withColumn("product_weight_g",when(col("product_weight_g") <= 0,None).otherwise(col("product_weight_g")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 135, Finished, Available, Finished, False)

#### Now i will replace null values with mean or median 

In [134]:
Products_df.select(
    mean(col("product_weight_g")).alias("mean"),
    stddev(col("product_weight_g")).alias("stddev"),
    min(col("product_weight_g")).alias("min"),
    max(col("product_weight_g")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_weight_g", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 136, Finished, Available, Finished, False)

+------------------+-----------------+---+-------+
|              mean|           stddev|min|    max|
+------------------+-----------------+---+-------+
|2276.7488845044772|4282.225203617538|2.0|40425.0|
+------------------+-----------------+---+-------+

Median: 700.0


#### Median is better because there are outliers

In [135]:
Products_df = Products_df.withColumn("product_weight_g",
    when(col("product_weight_g").isNull(), median_val )
    .otherwise(col("product_weight_g")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 137, Finished, Available, Finished, False)

In [136]:
display(Products_df)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 138, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f8a1e9e-514c-43b1-b8a9-2c2a62656710)

In [137]:
Products_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 139, Finished, Available, Finished, False)

root
 |-- Product_id: string (nullable = true)
 |-- product_category_name: string (nullable = false)
 |-- product_name_length: double (nullable = true)
 |-- product_description_length: double (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### I will do the same rules i applied to product_weight_g with product_length_cm , product_height_cm and product_width_cm

#### first column product length cm 

In [138]:
Products_df=Products_df.withColumn("product_length_cm",when(col("product_length_cm") <= 0,None).otherwise(col("product_length_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 140, Finished, Available, Finished, False)

In [139]:
Products_df.select(
    mean(col("product_length_cm")).alias("mean"),
    stddev(col("product_length_cm")).alias("stddev"),
    min(col("product_length_cm")).alias("min"),
    max(col("product_length_cm")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_length_cm", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 141, Finished, Available, Finished, False)

+-----------------+-----------------+---+-----+
|             mean|           stddev|min|  max|
+-----------------+-----------------+---+-----+
|30.81507784758263|16.91445805406598|7.0|105.0|
+-----------------+-----------------+---+-----+

Median: 25.0


In [140]:
Products_df = Products_df.withColumn("product_length_cm",
    when(col("product_length_cm").isNull(), median_val )
    .otherwise(col("product_length_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 142, Finished, Available, Finished, False)

#### Second Column product height cm

In [141]:
Products_df=Products_df.withColumn("product_height_cm",when(col("product_height_cm") <= 0,None).otherwise(col("product_height_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 143, Finished, Available, Finished, False)

In [142]:
Products_df.select(
    mean(col("product_height_cm")).alias("mean"),
    stddev(col("product_height_cm")).alias("stddev"),
    min(col("product_height_cm")).alias("min"),
    max(col("product_height_cm")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_height_cm", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 144, Finished, Available, Finished, False)

+------------------+------------------+---+-----+
|              mean|            stddev|min|  max|
+------------------+------------------+---+-----+
|16.937661234028347|13.637554061749531|2.0|105.0|
+------------------+------------------+---+-----+

Median: 13.0


In [143]:
Products_df = Products_df.withColumn("product_height_cm",
    when(col("product_height_cm").isNull(), median_val )
    .otherwise(col("product_height_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 145, Finished, Available, Finished, False)

#### Third column product width cm

In [144]:
Products_df=Products_df.withColumn("product_width_cm",when(col("product_width_cm") <= 0,None).otherwise(col("product_width_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 146, Finished, Available, Finished, False)

In [145]:
Products_df.select(
    mean(col("product_width_cm")).alias("mean"),
    stddev(col("product_width_cm")).alias("stddev"),
    min(col("product_width_cm")).alias("min"),
    max(col("product_width_cm")).alias("max")
).show()
median_val = Products_df.approxQuantile("product_width_cm", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 147, Finished, Available, Finished, False)

+------------------+------------------+---+-----+
|              mean|            stddev|min|  max|
+------------------+------------------+---+-----+
|23.196728277034204|12.079047453227838|6.0|118.0|
+------------------+------------------+---+-----+

Median: 20.0


In [146]:
Products_df = Products_df.withColumn("product_width_cm",
    when(col("product_width_cm").isNull(), median_val )
    .otherwise(col("product_width_cm")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 148, Finished, Available, Finished, False)

#### Column product category name english i will make the same rules like the column : product category name

In [147]:
Products_df=Products_df.fillna({"product_category_name_english":"unknown"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 149, Finished, Available, Finished, False)

In [148]:
Products_df = Products_df.withColumn("product_category_name_english",trim(col("product_category_name_english")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 150, Finished, Available, Finished, False)

In [149]:
Products_df=Products_df.withColumn("product_category_name_english",when(col("product_category_name_english") == "","unknwon").otherwise(col("product_category_name_english"))) 

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 151, Finished, Available, Finished, False)

In [150]:
Products_df=Products_df.withColumn("product_category_name_english",lower(col("product_category_name_english")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 152, Finished, Available, Finished, False)

In [151]:
display(Products_df)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 153, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2005822f-b014-499a-aa5e-af0ebc0c47b6)

#### Now i will save this table in silver schema 

In [152]:
Products_df = Products_df.withColumn("silver_loaded_date", current_timestamp())
Products_df = Products_df.withColumn("silver_source_table", lit("bronze.products"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 154, Finished, Available, Finished, False)

In [153]:
display(Products_df)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 155, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 44a2b998-587c-48b8-beb3-3678e5e16d1a)

In [157]:
Products_df=Products_df.withColumn("product_name_length",col("product_name_length").cast("int"))
Products_df=Products_df.withColumn("product_description_length",col("product_description_length").cast("int"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 159, Finished, Available, Finished, False)

In [158]:
Products_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 160, Finished, Available, Finished, False)

root
 |-- Product_id: string (nullable = true)
 |-- product_category_name: string (nullable = false)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- product_category_name_english: string (nullable = false)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)
 |-- silver_loaded_date: timestamp (nullable = false)
 |-- silver_source_table: string (nullable = false)



In [159]:
Products_df.write.format("delta").mode("append").saveAsTable("silver.Products")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 161, Finished, Available, Finished, False)

### Now Sellers Table 

In [161]:
Sellers_df=spark.read.table("Olist_Lakehouse.bronze.sellers")
display(Sellers_df)

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 163, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45053a31-21a1-409a-9ff9-baee74d7133f)

#### Seller_id i will do the same steps i made with all primary keys

In [163]:
Sellers_df=Sellers_df.filter(col("seller_id").isNotNull())

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 165, Finished, Available, Finished, False)

In [164]:
Sellers_df=Sellers_df.withColumn("seller_id",trim(col("seller_id")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 166, Finished, Available, Finished, False)

In [165]:
Sellers_df=Sellers_df.filter(col("seller_id") != "")

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 167, Finished, Available, Finished, False)

In [166]:
Sellers_df=Sellers_df.dropDuplicates(["seller_id"])

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 168, Finished, Available, Finished, False)

#### Column zip_code same steps like before 

In [167]:
Sellers_df=Sellers_df.fillna({"seller_zip_code_prefix":0})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 169, Finished, Available, Finished, False)

In [168]:
Sellers_df=Sellers_df.withColumn("seller_zip_code_prefix",col("seller_zip_code_prefix").cast("string"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 170, Finished, Available, Finished, False)

In [169]:
Sellers_df=Sellers_df.withColumn("seller_zip_code_prefix",trim(col("seller_zip_code_prefix")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 171, Finished, Available, Finished, False)

In [170]:
Sellers_df=Sellers_df.withColumn("seller_zip_code_prefix",lpad(col("seller_zip_code_prefix"),5,"0"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 172, Finished, Available, Finished, False)

#### Column seller_city Same Steps as Before 



In [171]:
Sellers_df=Sellers_df.fillna({"seller_city":"unknown"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 173, Finished, Available, Finished, False)

In [172]:
Sellers_df=Sellers_df.withColumn("seller_city",trim(col("seller_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 174, Finished, Available, Finished, False)

In [173]:
Sellers_df=Sellers_df.withColumn("seller_city",when(col("seller_city") == "","unknown").otherwise(col("seller_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 175, Finished, Available, Finished, False)

In [174]:
Sellers_df=Sellers_df.withColumn("seller_city",lower(col("seller_city")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 176, Finished, Available, Finished, False)

### Column Seller_state Same steps as before

In [175]:
Sellers_df=Sellers_df.fillna({"seller_state":"UN"})

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 177, Finished, Available, Finished, False)

In [176]:
Sellers_df=Sellers_df.withColumn("seller_state",trim(col("seller_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 178, Finished, Available, Finished, False)

In [177]:
Sellers_df=Sellers_df.withColumn("seller_state",when(col("seller_state") == "" ,"UN").otherwise(upper(col("seller_state"))))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 179, Finished, Available, Finished, False)

In [178]:
Sellers_df=Sellers_df.withColumn("seller_state",when(length(col("seller_state")) != 2 ,"UN").otherwise(col("seller_state")))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 180, Finished, Available, Finished, False)

In [179]:
Sellers_df.printSchema()

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 181, Finished, Available, Finished, False)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = false)
 |-- seller_state: string (nullable = false)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



In [180]:
Sellers_df = Sellers_df.withColumn("silver_loaded_date", current_timestamp())
Sellers_df = Sellers_df.withColumn("silver_source_table", lit("bronze.sellers"))

StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 182, Finished, Available, Finished, False)

In [181]:
Sellers_df.write.format("delta").mode("append").saveAsTable("silver.sellers")


StatementMeta(, d3f17566-c2f4-4881-b4a7-e179cabad3d5, 183, Finished, Available, Finished, False)

### Now geolocation Table

In [1]:
Geolocation_df=spark.read.table("Olist_Lakehouse.bronze.geolocation")
display(Geolocation_df)

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73129571-eadc-460c-8551-ba23c5214025)

#### Column geolocation_zip_code_prefix

#### Same steps as before 

In [2]:
Geolocation_df=Geolocation_df.fillna({"geolocation_zip_code_prefix":0})

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 4, Finished, Available, Finished, False)

In [5]:
Geolocation_df=Geolocation_df.withColumn("geolocation_zip_code_prefix",col("geolocation_zip_code_prefix").cast("string"))

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 7, Finished, Available, Finished, False)

In [6]:
Geolocation_df=Geolocation_df.withColumn("geolocation_zip_code_prefix",trim(col("geolocation_zip_code_prefix")))

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 8, Finished, Available, Finished, False)

In [7]:
Geolocation_df=Geolocation_df.withColumn("geolocation_zip_code_prefix",lpad("geolocation_zip_code_prefix",5,"0"))

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 9, Finished, Available, Finished, False)

In [8]:
Geolocation_df=Geolocation_df.dropDuplicates(["geolocation_zip_code_prefix"])

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 10, Finished, Available, Finished, False)

In [10]:
display(Geolocation_df.filter(col("geolocation_zip_code_prefix")== "00000"))

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ded71ea-14bd-4b60-8847-cd5687241bde)

In [11]:
Geolocation_df = Geolocation_df.withColumn("silver_loaded_date", current_timestamp())
Geolocation_df = Geolocation_df.withColumn("silver_source_table", lit("bronze.geolocation"))

StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 13, Finished, Available, Finished, False)

In [12]:
Geolocation_df.write.format("delta").mode("append").saveAsTable("silver.geolocation")


StatementMeta(, 479eb8b1-739a-45cf-9c87-ccb43fce6ad8, 14, Finished, Available, Finished, False)

### Now Order_reviews Table

In [1]:
Order_Reviews_df=spark.read.table("Olist_Lakehouse.bronze.order_reviews")
display(Order_Reviews_df)

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00863c5d-8acb-4016-a1f7-1501d55f2038)

#### Column Review_id

#### The same steps like each primary key 

In [5]:
Order_Reviews_df= Order_Reviews_df.filter(col("review_id").isNotNull())

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 7, Finished, Available, Finished, False)

In [6]:
Order_Reviews_df= Order_Reviews_df.withColumn("review_id",trim(col("review_id"))) 

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 8, Finished, Available, Finished, False)

In [7]:
Order_Reviews_df= Order_Reviews_df.filter(col("review_id") != "")

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 9, Finished, Available, Finished, False)

In [8]:
Order_Reviews_df= Order_Reviews_df.dropDuplicates(["review_id"])

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 10, Finished, Available, Finished, False)

#### Column Order_id

#### if order_id is null i will remove the entire row 

In [10]:
Order_Reviews_df= Order_Reviews_df.filter(col("order_id").isNotNull())

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 12, Finished, Available, Finished, False)

#### I will Trim order_id

In [11]:
Order_Reviews_df= Order_Reviews_df.withColumn("order_id",trim(col("order_id"))) 

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 13, Finished, Available, Finished, False)

#### i will remove values like ""

In [12]:
Order_Reviews_df= Order_Reviews_df.filter(col("order_id") != "")

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 14, Finished, Available, Finished, False)

#### Column Review Score

In [13]:
Order_Reviews_df.printSchema()

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 15, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



In [14]:
Order_Reviews_df= Order_Reviews_df.filter((col("review_score") >=1) & (col("review_score") <=5) | col("review_score").isNull())

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 16, Finished, Available, Finished, False)

#### Column review_comment_title

#### I will trim it and replace "" with None 

In [16]:
Order_Reviews_df=Order_Reviews_df.withColumn("review_comment_title",trim(col("review_comment_title")))

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 18, Finished, Available, Finished, False)

In [18]:
Order_Reviews_df=Order_Reviews_df.withColumn("review_comment_title",when(col("review_comment_title") == "",None).otherwise(col("review_comment_title")))

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 20, Finished, Available, Finished, False)

#### Column review_comment_message

#### Same as Review_Comment_title

In [19]:
Order_Reviews_df=Order_Reviews_df.withColumn("review_comment_message",trim(col("review_comment_message")))

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 21, Finished, Available, Finished, False)

In [20]:
Order_Reviews_df=Order_Reviews_df.withColumn("review_comment_message",when(col("review_comment_message") == "",None).otherwise(col("review_comment_message")))

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 22, Finished, Available, Finished, False)

#### Column review Creation date

In [21]:
Order_Reviews_df.printSchema()

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 23, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### I need only dates bigger than 1-1-2016

In [22]:
Order_Reviews_df = Order_Reviews_df.filter(col("review_creation_date") >= "2016-01-01")


StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 24, Finished, Available, Finished, False)

#### Column review_answer_timestamp

#### i will drop row if this date before creation date 

In [23]:
Order_Reviews_df=Order_Reviews_df.withColumn("review_date_error",when((col("review_answer_timestamp").isNotNull()) & (col("review_answer_timestamp") < col("review_creation_date")),True).otherwise(False) )

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 25, Finished, Available, Finished, False)

In [24]:
Order_Reviews_df=Order_Reviews_df.filter(col("review_date_error") == False)

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 26, Finished, Available, Finished, False)

In [25]:
Order_Reviews_df = Order_Reviews_df.withColumn("silver_loaded_date", current_timestamp())
Order_Reviews_df = Order_Reviews_df.withColumn("silver_source_table", lit("bronze.order_reviews"))

StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 27, Finished, Available, Finished, False)

In [26]:
Order_Reviews_df.write.format("delta").mode("append").saveAsTable("silver.order_reviews")


StatementMeta(, 5e596c9c-2cad-422e-b43c-f4c84bb407c9, 28, Finished, Available, Finished, False)

### Table order_payments

In [2]:
Order_Payments_df=spark.read.table("Olist_Lakehouse.bronze.order_payments")
display(Order_Payments_df)

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 691bb674-236d-42ae-97b3-5c5041372826)

#### Composite primary key between (order_id , payment_sequential)

#### So each one of them could have duplicates values but two of them must not have   

#### First i will drop rows if one of them has nulls

In [3]:
Order_Payments_df=Order_Payments_df.filter(col("order_id").isNotNull())

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 5, Finished, Available, Finished, False)

In [4]:
Order_Payments_df=Order_Payments_df.filter(col("payment_sequential").isNotNull())

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 6, Finished, Available, Finished, False)

In [5]:
Order_Payments_df.printSchema()

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 7, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### I will trim order_id

In [6]:
Order_Payments_df=Order_Payments_df.withColumn("order_id",trim(col("order_id")))

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 8, Finished, Available, Finished, False)

In [7]:
Order_Payments_df=Order_Payments_df.filter(col("order_id") != "")

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 9, Finished, Available, Finished, False)

#### i need distinct values of order_id and payment_sequential 

In [8]:
Order_Payments_df=Order_Payments_df.dropDuplicates(["order_id","payment_sequential"])

StatementMeta(, b6ac90cb-d89b-4f4c-9eeb-ecaeefde2efe, 10, Finished, Available, Finished, False)

#### Column Payment_Type

#### I will fill null values with "Not Defined"

In [3]:
Order_Payments_df=Order_Payments_df.fillna({"payment_type":"Not Defined"})

StatementMeta(, e91a5609-652c-43ce-9b8a-99a71a5dc634, 5, Finished, Available, Finished, False)

#### I will trim it 

In [6]:
Order_Payments_df=Order_Payments_df.withColumn("payment_type",trim(col("payment_type")))

StatementMeta(, e91a5609-652c-43ce-9b8a-99a71a5dc634, 8, Finished, Available, Finished, False)

#### I will replace this value "" with "Not Defined"

In [7]:
Order_Payments_df=Order_Payments_df.withColumn("payment_type",when(col("payment_type") == "","Not Defined").otherwise(col("payment_type")))

StatementMeta(, e91a5609-652c-43ce-9b8a-99a71a5dc634, 9, Finished, Available, Finished, False)

#### Column payment_installments

#### if its value is null i will replace it with 1 like he pays on one time and if its value less than 1 or zero i will replace it with one


In [4]:
Order_Payments_df=Order_Payments_df.withColumn("payment_installments",when(col("payment_installments").isNull(),1).when(col("payment_installments") <=0 ,1).otherwise(col("payment_installments")))

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 6, Finished, Available, Finished, False)

#### Column payment_value

In [6]:
Order_Payments_df.select(
    mean(col("payment_value")).alias("mean"),
    stddev(col("payment_value")).alias("stddev"),
    min(col("payment_value")).alias("min"),
    max(col("payment_value")).alias("max")
).show()
median_val = Order_Payments_df.approxQuantile("payment_value", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 8, Finished, Available, Finished, False)

+-----+----------------+-----+------+
| mean|          stddev|  min|   max|
+-----+----------------+-----+------+
|84.08|48.2761835691265|40.95|136.23|
+-----+----------------+-----+------+

Median: 75.06


#### I will replace null values with median 

In [7]:
Order_Payments_df = Order_Payments_df.withColumn("payment_value",
    when(col("payment_value").isNull(), median_val )
    .otherwise(col("payment_value")))

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 9, Finished, Available, Finished, False)

In [9]:
Order_Payments_df=Order_Payments_df.filter(col("payment_value") >= 0)

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 11, Finished, Available, Finished, False)

In [10]:
Order_Payments_df = Order_Payments_df.withColumn("silver_loaded_date", current_timestamp())
Order_Payments_df = Order_Payments_df.withColumn("silver_source_table", lit("bronze.order_payments"))

StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 12, Finished, Available, Finished, False)

In [11]:
Order_Payments_df.write.format("delta").mode("append").saveAsTable("silver.order_payments")


StatementMeta(, b2a7126a-48f7-4892-8798-1825519ad90c, 13, Finished, Available, Finished, False)

### Table order_items

In [1]:
Order_items_df=spark.read.table("Olist_Lakehouse.bronze.order_items")
display(Order_items_df)

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 435d35a3-f963-4c4a-9cd6-6e5c6f0cc388)

In [2]:
Order_items_df.printSchema()

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 4, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### here also we have composite key (order_id,order_item_id) so each one of them could have duplicates but both of them must not have 

In [5]:
Order_items_df=Order_items_df.filter(col("order_id").isNotNull())

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 7, Finished, Available, Finished, False)

In [6]:
Order_items_df=Order_items_df.filter(col("order_item_id").isNotNull())

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 8, Finished, Available, Finished, False)

#### I will trim order_id

In [7]:
Order_items_df=Order_items_df.withColumn("order_id",trim(col("order_id")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 9, Finished, Available, Finished, False)

#### i will drop order_id with value ""

In [8]:
Order_items_df=Order_items_df.filter(col("order_id") != "")

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 10, Finished, Available, Finished, False)

#### I will drop duplicate values of (order_id,order_item_id)

In [9]:
Order_items_df=Order_items_df.dropDuplicates(["order_id","order_item_id"])

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 11, Finished, Available, Finished, False)

#### Column product_id 

#### if its value is null i will drop it and i will trim it and if its value is "" i will drop it

In [10]:
Order_items_df=Order_items_df.filter(col("product_id").isNotNull())

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 12, Finished, Available, Finished, False)

In [11]:
Order_items_df=Order_items_df.withColumn("product_id",trim(col("product_id")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 13, Finished, Available, Finished, False)

In [12]:
Order_items_df=Order_items_df.filter(col("product_id") != "")

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 14, Finished, Available, Finished, False)

#### column seller_id 

#### I will do the same steps as product_id

In [13]:
Order_items_df=Order_items_df.filter(col("seller_id").isNotNull())

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 15, Finished, Available, Finished, False)

In [14]:
Order_items_df=Order_items_df.withColumn("seller_id",trim(col("seller_id")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 16, Finished, Available, Finished, False)

In [15]:
Order_items_df=Order_items_df.filter(col("seller_id") != "")

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 17, Finished, Available, Finished, False)

#### Column shipping_limit_date

#### i will keep only dates from 1-1-2016 to today 

In [16]:
Order_items_df=Order_items_df.filter(col("shipping_limit_date").isNull() | (col("shipping_limit_date") <= current_timestamp()))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 18, Finished, Available, Finished, False)

In [17]:
Order_items_df=Order_items_df.filter(col("shipping_limit_date").isNull() |  (col("shipping_limit_date") > "2016-01-01"))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 19, Finished, Available, Finished, False)

#### Column Price

In [18]:
Order_items_df=Order_items_df.withColumn("price",when(col("price") <= 0,None ).otherwise(col("price")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 20, Finished, Available, Finished, False)

#### i will replace Null Values by median

In [19]:
Order_items_df.select(
    mean(col("price")).alias("mean"),
    stddev(col("price")).alias("stddev"),
    min(col("price")).alias("min"),
    max(col("price")).alias("max")
).show()
median_val = Order_items_df.approxQuantile("price", [0.5], 0.01)[0]
print(f"Median: {median_val}")

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 21, Finished, Available, Finished, False)

+-----+----------------+----+----+
| mean|          stddev| min| max|
+-----+----------------+----+----+
|44.56|8.72466389037423|32.9|59.5|
+-----+----------------+----+----+

Median: 44.99


In [20]:
Order_items_df = Order_items_df.withColumn("price",
    when(col("price").isNull(), median_val )
    .otherwise(col("price")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 22, Finished, Available, Finished, False)

#### Column freight_value

#### if its value is null i will replace it with 0 as free shipping and if its value is negative i will replace it with zero also 

In [22]:
Order_items_df=Order_items_df.withColumn("freight_value",when(col("freight_value").isNull(),0).when(col("freight_value")<0,0).otherwise(col("freight_value")))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 24, Finished, Available, Finished, False)

In [23]:
Order_items_df = Order_items_df.withColumn("silver_loaded_date", current_timestamp())
Order_items_df = Order_items_df.withColumn("silver_source_table", lit("bronze.order_items"))

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 25, Finished, Available, Finished, False)

In [24]:
Order_items_df.write.format("delta").mode("append").saveAsTable("silver.order_items")

StatementMeta(, 610315c2-74d6-47c1-8449-ae0d91fa130b, 26, Finished, Available, Finished, False)

### Orders Table

In [54]:
Orders_df=spark.read.table("Olist_Lakehouse.bronze.orders")
display(Orders_df)

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7ae4843-3953-4f2c-94f2-61dcf3a67b6a)

#### Column order_id same steps as before with primary keys

In [55]:
Orders_df=Orders_df.filter(col("order_id").isNotNull())

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 57, Finished, Available, Finished, False)

In [56]:
Orders_df=Orders_df.withColumn("order_id",trim(col("order_id")))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 58, Finished, Available, Finished, False)

In [57]:
Orders_df=Orders_df.filter(col("order_id") != "")

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 59, Finished, Available, Finished, False)

In [58]:
Orders_df=Orders_df.dropDuplicates(["order_id"])

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 60, Finished, Available, Finished, False)

#### Column Customer_id

#### I will drop null values and trim it and drop values with ""

In [59]:
Orders_df=Orders_df.filter(col("customer_id").isNotNull())

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 61, Finished, Available, Finished, False)

In [60]:
Orders_df=Orders_df.withColumn("customer_id",trim(col("customer_id")))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 62, Finished, Available, Finished, False)

In [61]:
Orders_df=Orders_df.filter(col("customer_id") != "")

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 63, Finished, Available, Finished, False)

#### Column order_status

#### I will replace null values with "unknown"

In [62]:
Orders_df=Orders_df.fillna({"order_status":"unknown"})

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 64, Finished, Available, Finished, False)

#### I will trim it 

In [63]:
Orders_df=Orders_df.withColumn("order_status",trim(col("order_status")))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 65, Finished, Available, Finished, False)

#### I will replace "" with "unknown"

In [64]:
Orders_df=Orders_df.withColumn("order_status",when(col("order_status") == "" , "unknown").otherwise(col("order_status")))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 66, Finished, Available, Finished, False)

#### I will make its values lower case 

In [65]:
Orders_df=Orders_df.withColumn("order_status",lower(col("order_status")))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 67, Finished, Available, Finished, False)

#### Column order_purchase_timestamp

#### I will drop the row if order_purchase_timestamp is null because it must have a value 

In [66]:
Orders_df=Orders_df.filter(col("order_purchase_timestamp").isNotNull())

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 68, Finished, Available, Finished, False)

#### The value must be bigger than 1-1-2016 and less than current date 

In [67]:
Orders_df=Orders_df.filter(col("order_purchase_timestamp") >= "2016-01-01")
Orders_df=Orders_df.filter(col("order_purchase_timestamp") <= current_timestamp())

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 69, Finished, Available, Finished, False)

#### all other date column i have to check only if they are timestamp because if i make a check range it will drop rows because there are null values 

In [68]:
Orders_df.printSchema()

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 70, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = false)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- Source_Table: string (nullable = true)



#### here i make column to check the sequence of operations is right 

In [69]:
Orders_df = Orders_df.withColumn("date_logic_error",
    when(col("order_delivered_customer_date") < col("order_purchase_timestamp"), True)
    .when(col("order_delivered_carrier_date") < col("order_purchase_timestamp"), True)
    .when(col("order_approved_at") < col("order_purchase_timestamp"), True)
    .otherwise(False))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 71, Finished, Available, Finished, False)

In [70]:
Orders_df=Orders_df.filter(col("date_logic_error") == False)

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 72, Finished, Available, Finished, False)

In [71]:
Orders_df = Orders_df.withColumn("silver_loaded_date", current_timestamp())
Orders_df = Orders_df.withColumn("silver_source_table", lit("bronze.orders"))

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 73, Finished, Available, Finished, False)

In [72]:
Orders_df.write.format("delta").mode("append").saveAsTable("silver.orders")

StatementMeta(, c88f760a-e848-4ea5-8510-ea0ceb596e7b, 74, Finished, Available, Finished, False)